# 🚀 Fine-tuning de Modelo para Manuais Boeing 737

Este notebook treina um modelo pequeno para responder **apenas** com base nos manuais fornecidos.

## ⚠️ Pré-requisitos

### Opção 1: TinyLlama (Recomendado - sem login)
- Não precisa de login no Hugging Face
- Mais rápido para testar

### Opção 2: Llama-3.2 (Precisa de login)
1. Crie conta em [huggingface.co](https://huggingface.co)
2. Vá em [Settings > Access Tokens](https://huggingface.co/settings/tokens)
3. Clique em "New token"
4. Nome: `colab-finetuning`
5. Tipo: `Read`
6. Copie o token
7. No Colab, execute a célula de login e cole o token

## Como usar no Google Colab
1. Faça upload do arquivo `dados_treino.json` (gerado no seu computador)
2. Execute todas as células
3. Baixe o modelo treinado no final

## 1. Instalação de Dependências

In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes==0.41.3 trl\n\nimport torch\nprint(f"PyTorch: {torch.__version__}")\nprint(f"CUDA disponível: {torch.cuda.is_available()}")\nprint(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

## 2. Upload e Preparação dos Dados

In [ ]:
from google.colab import files
import json

# Upload do arquivo dados_treino.json
print("Faça upload do arquivo 'dados_treino.json':")
uploaded = files.upload()

# Carregar dados
with open('dados_treino.json', 'r', encoding='utf-8') as f:
    dados = json.load(f)

print(f"✓ {len(dados)} exemplos carregados")
print(f"\nExemplo:")
print(json.dumps(dados[0], indent=2, ensure_ascii=False))

In [ ]:
# Preparar no formato de instrução (Alpaca format)
def formatar_instrucao(exemplo):
    return {
        "instruction": exemplo["pergunta"],
        "input": "",  # Não usado, mas mantido para compatibilidade
        "output": exemplo["resposta"],
        "text": f"""### Instrução:
{exemplo['pergunta']}

### Resposta:
{exemplo['resposta']}"""
    }

dados_formatados = [formatar_instrucao(d) for d in dados]
print(f"✓ {len(dados_formatados)} exemplos formatados")

# Dividir em treino/validação
from sklearn.model_selection import train_test_split
treino, validacao = train_test_split(dados_formatados, test_size=0.1, random_state=42)
print(f"✓ Treino: {len(treino)} | Validação: {len(validacao)}")

## 3. Configurar Modelo e Tokenizer

In [ ]:
# Login no Hugging Face (necessário para modelos Llama)
from huggingface_hub import login
import os

# Opção 1: Token como variável de ambiente (mais seguro)
if "HF_TOKEN" in os.environ:
    login(token=os.environ["HF_TOKEN"])
    print("✓ Login via variável de ambiente")
else:
    # Opção 2: Login interativo
    print("Cole seu token do Hugging Face:")
    print("Obtenha em: https://huggingface.co/settings/tokens")
    print("(Marque 'Read' como permissão)")
    login()

print("✓ Login realizado com sucesso!")

## 3. Configurar Modelo e Tokenizer

**Opção A**: Llama-3.2-1B (precisa de login no Hugging Face)
**Opção B**: TinyLlama-1.1B (não precisa de login)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

# OPÇÃO A: TinyLlama (não precisa de login) - RECOMENDADO PARA TESTE RÁPIDO
modelo_nome = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# OPÇÃO B: Llama-3.2-1B (precisa de login, comentado)
# modelo_nome = "meta-llama/Llama-3.2-1B-Instruct"

print(f"Carregando: {modelo_nome}")

tokenizer = AutoTokenizer.from_pretrained(modelo_nome)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    modelo_nome,
    torch_dtype=torch.float16,
    device_map="auto",
    load_in_8bit=True
)

print(f"✓ Modelo carregado: {model.num_parameters():,} parâmetros")

In [ ]:
# Configurar LoRA (Low-Rank Adaptation) - treina só 0.1% dos parâmetros
lora_config = LoraConfig(
    r=16,  # Rank
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

print("✓ LoRA configurado")
model.print_trainable_parameters()

## 4. Preparar Dataset

In [ ]:
from datasets import Dataset

def tokenizar(exemplo):
    resultado = tokenizer(
        exemplo["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )
    resultado["labels"] = resultado["input_ids"].copy()
    return resultado

dataset_treino = Dataset.from_list(treino).map(tokenizar, batched=False)
dataset_validacao = Dataset.from_list(validacao).map(tokenizar, batched=False)

print(f"✓ Datasets prontos")
print(f"  Treino: {len(dataset_treino)}")
  Validação: {len(dataset_validacao)}

## 5. Treinamento

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./modelo_b737",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=50,
    save_steps=500,
    eval_steps=500,
    save_total_limit=2,
    warmup_steps=100,
    fp16=True,
    push_to_hub=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_treino,
    eval_dataset=dataset_validacao,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

print("🚀 Iniciando treinamento...")
trainer.train()

## 6. Salvar Modelo

In [ ]:
# Salvar modelo LoRA
model.save_pretrained("./modelo_b737_final")
tokenizer.save_pretrained("./modelo_b737_final")

print("✓ Modelo salvo em ./modelo_b737_final")

# Compactar para download
!zip -r modelo_b737_final.zip modelo_b737_final/

print("\n📦 Modelo compactado! Faça download do arquivo modelo_b737_final.zip")

## 7. Testar o Modelo

In [ ]:
def testar(pergunta):
    prompt = f"""### Instrução:
{pergunta}

### Resposta:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.3,
        do_sample=True
    )
    resposta = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return resposta.split("### Resposta:")[-1].strip()

# Testes
testes = [
    "Quantos sistemas hidráulicos tem o Boeing 737?",
    "Quantos Flight Control Computers tem o Boeing 737?",
    "Qual é a cor da pintura do Boeing 737?"  # Deve recusar
]

for t in testes:
    print(f"\n❓ {t}")
    print(f"💬 {testar(t)}")

## 8. Download do Modelo

In [ ]:
from google.colab import files

# Fazer download do modelo treinado
files.download('modelo_b737_final.zip')

print("✅ Download iniciado! Descompacte o arquivo na pasta do seu projeto.")